# Session 9: Synthetic Data Generation and RAG Evaluation with LangSmith

In the following notebook we'll explore a use-case for RAGAS' synthetic testset generation workflow, and use it to evaluate and iterate on a RAG pipeline with LangSmith!

**Learning Objectives:**
- Understand Ragas' knowledge graph-based synthetic data generation workflow
- Generate synthetic test sets with different query synthesizer types
- Load synthetic data into LangSmith for evaluation
- Evaluate a RAG chain using LangSmith evaluators
- Iterate on RAG pipeline parameters and measure the impact

## Table of Contents:

- **Breakout Room #1:** Synthetic Data Generation with Ragas
  - Task 1: Dependencies and API Keys
  - Task 2: Data Preparation and Knowledge Graph Construction
  - Task 3: Generating Synthetic Test Data
  - Question #1 & Question #2
  - 🏗️ Activity #1: Custom Query Distribution

- **Breakout Room #2:** RAG Evaluation with LangSmith
  - Task 4: LangSmith Dataset Setup
  - Task 5: Building a Basic RAG Chain
  - Task 6: Evaluating with LangSmith
  - Task 7: Modifying the Pipeline and Re-Evaluating
  - Question #3 & Question #4
  - 🏗️ Activity #2: Analyze Evaluation Results

---
# 🤝 Breakout Room #1
## Synthetic Data Generation with Ragas

## Task 1: Dependencies and API Keys

We'll need to install a number of API keys and dependencies, since we'll be leveraging a number of great technologies for this pipeline!

1. OpenAI's endpoints to handle the Synthetic Data Generation
2. OpenAI's Endpoints for our RAG pipeline and LangSmith evaluation
3. QDrant as our vectorstore
4. LangSmith for our evaluation coordinator!

Let's install and provide all the required information below!

## Dependencies and API Keys:

### NLTK Import

To prevent errors that may occur based on OS - we'll import NLTK and download the needed packages to ensure correct handling of data.

In [1]:
import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/swarabankhele/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/swarabankhele/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.


True

In [2]:
import os
import getpass

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangChain API Key:")

We'll also want to set a project name to make things easier for ourselves.

In [3]:
from uuid import uuid4

os.environ["LANGCHAIN_PROJECT"] = f"AIM - SDG - {uuid4().hex[0:8]}"

OpenAI's API Key!

In [4]:
os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key:")

## Generating Synthetic Test Data

We wil be using Ragas to build out a set of synthetic test questions, references, and reference contexts. This is useful because it will allow us to find out how our system is performing.

> NOTE: Ragas is best suited for finding *directional* changes in your LLM-based systems. The absolute scores aren't comparable in a vacuum.

### Data Preparation

We'll prepare our data using two complementary guides — a Health & Wellness Guide covering exercise, nutrition, sleep, and stress management, and a Mental Health & Psychology Handbook covering mental health conditions, therapeutic approaches, resilience, and daily mental health practices. The topical overlap between documents helps RAGAS build rich cross-document relationships in the knowledge graph.

Next, let's load our data into a familiar LangChain format using the `TextLoader`.

In [5]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

loader = DirectoryLoader("data/", glob="*.txt", loader_cls=TextLoader)
docs = loader.load()
print(f"Loaded {len(docs)} documents: {[d.metadata['source'] for d in docs]}")

Loaded 2 documents: ['data/MentalHealthGuide.txt', 'data/HealthWellnessGuide.txt']


### Knowledge Graph Based Synthetic Generation

Ragas uses a knowledge graph based approach to create data. This is extremely useful as it allows us to create complex queries rather simply. The additional testset complexity allows us to evaluate larger problems more effectively, as systems tend to be very strong on simple evaluation tasks.

Let's start by defining our `generator_llm` (which will generate our questions, summaries, and more), and our `generator_embeddings` which will be useful in building our graph.

### Unrolled SDG

In [52]:
from ragas.llms import llm_factory
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import OpenAIEmbeddings

# Ragas 0.2.10 llm_factory: model, run_config, default_headers, base_url only (no client)
generator_llm = llm_factory("gpt-4.1-nano")
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

Next, we're going to instantiate our Knowledge Graph.

This graph will contain N number of nodes that have M number of relationships. These nodes and relationships (AKA "edges") will define our knowledge graph and be used later to construct relevant questions and responses.

In [9]:
from ragas.testset.graph import KnowledgeGraph

kg = KnowledgeGraph()
kg

KnowledgeGraph(nodes: 0, relationships: 0)

The first step we're going to take is to simply insert each of our full documents into the graph. This will provide a base that we can apply transformations to.

In [10]:
from ragas.testset.graph import Node, NodeType

for doc in docs:
    kg.nodes.append(
        Node(
            type=NodeType.DOCUMENT,
            properties={"page_content": doc.page_content, "document_metadata": doc.metadata}
        )
    )
kg

KnowledgeGraph(nodes: 2, relationships: 0)

Now, we'll apply the *default* transformations to our knowledge graph. This will take the nodes currently on the graph and transform them based on a set of [default transformations](https://docs.ragas.io/en/latest/references/transforms/#ragas.testset.transforms.default_transforms).

These default transformations are dependent on the corpus length, in our case:

- Producing Summaries -> produces summaries of the documents
- Extracting Headlines -> finding the overall headline for the document
- Theme Extractor -> extracts broad themes about the documents

It then uses cosine-similarity and heuristics between the embeddings of the above transformations to construct relationships between the nodes.

In [22]:
from ragas.testset.transforms import default_transforms, apply_transforms

transformer_llm = generator_llm
embedding_model = generator_embeddings

default_transforms = default_transforms(documents=docs, llm=transformer_llm, embedding_model=embedding_model)
apply_transforms(kg, default_transforms)
kg

Applying HeadlinesExtractor:   0%|          | 0/2 [00:00<?, ?it/s]

Property 'headlines' already exists in node '9e8ae0'. Skipping!
Property 'headlines' already exists in node 'fd3f08'. Skipping!


Applying HeadlineSplitter:   0%|          | 0/9 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node


Applying SummaryExtractor:   0%|          | 0/2 [00:00<?, ?it/s]

...

Applying CustomNodeFilter:   0%|          | 0/14 [00:00<?, ?it/s]

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/30 [00:00<?, ?it/s]

Property 'summary_embedding' already exists in node 'fd3f08'. Skipping!
Property 'summary_embedding' already exists in node '9e8ae0'. Skipping!
Property 'entities' already exists in node 'b56797'. Skipping!
Property 'entities' already exists in node 'd80502'. Skipping!
Property 'themes' already exists in node '3ef307'. Skipping!
Property 'themes' already exists in node 'b56797'. Skipping!
Property 'themes' already exists in node '0dea25'. Skipping!
Property 'themes' already exists in node 'd5d453'. Skipping!
Property 'entities' already exists in node '8f65b9'. Skipping!
Property 'entities' already exists in node 'b69f9d'. Skipping!
Property 'themes' already exists in node 'b69f9d'. Skipping!
Property 'themes' already exists in node 'd80502'. Skipping!
Property 'entities' already exists in node '3ef307'. Skipping!
Property 'themes' already exists in node '8f65b9'. Skipping!
Property 'entities' already exists in node '0dea25'. Skipping!
Property 'entities' already exists in node 'd5d453'

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

KnowledgeGraph(nodes: 16, relationships: 36)

We can save and load our knowledge graphs as follows.

In [23]:
kg.save("usecase_data_kg.json")
usecase_data_kg = KnowledgeGraph.load("usecase_data_kg.json")
usecase_data_kg

KnowledgeGraph(nodes: 16, relationships: 36)

Using our knowledge graph, we can construct a "test set generator" - which will allow us to create queries.

In [24]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=embedding_model, knowledge_graph=usecase_data_kg)

However, we'd like to be able to define the kinds of queries we're generating - which is made simple by Ragas having pre-created a number of different "QuerySynthesizer"s.

Each of these Synthetsizers is going to tackle a separate kind of query which will be generated from a scenario and a persona.

In essence, Ragas will use an LLM to generate a persona of someone who would interact with the data - and then use a scenario to construct a question from that data and persona.

In [25]:
from ragas.testset.synthesizers import default_query_distribution, SingleHopSpecificQuerySynthesizer, MultiHopAbstractQuerySynthesizer, MultiHopSpecificQuerySynthesizer

query_distribution = [
        (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.5),
        (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0.25),
        (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0.25),
]

## ❓ Question #1:

What are the three types of query synthesizers doing? Describe each one in simple terms.

##### Answer:
1. SingleHopSpecificQuerySynthesizer: This generates sungle-hop question, the answer can be found in one document or knowledge-graph node. They are specific because they target concreete facts in that source. (e.g. “What are the healthy fats mentioned in the content?”)

2. MultiHopSpecificQuerySynthesizer: This generates multi-hop questions requiring information from multiple sources. They are specific because they ask for precise, factual answers abd often refer to concrete entities. (e.g. What do chapter 9 and 16 say about sleep and social connections?) The system must retrieve from several chunks and combine specific facts into one answer.

3. MultiHopAbstractQuerySynthesizer: This also genarted multi-hopquestions: the answer requires combining information from multiple documents or nodes. They are abstarct because they focus on concepts and higher-levl understanding rather than exact quotes or details. For example: “How can mindfulness and meditation techniques support mental health?” or “How can social connections support well-being?” The RAG system has to pull from several sources and synthesize a conceptual answer.

Finally, we can use our `TestSetGenerator` to generate our testset!

In [27]:
testset = generator.generate(testset_size=10, query_distribution=query_distribution)
testset.to_pandas()

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/11 [00:00<?, ?it/s]

,user_input,reference_contexts,reference,synthesizer_name
0,What does the United States' approach to menta...,[The Mental Health and Psychology Handbook A P...,The context provided does not include specific...,single_hop_specifc_query_synthesizer
1,What is the University of Massachusetts Medica...,[PART 2: THERAPEUTIC APPROACHES Chapter 4: Cog...,The University of Massachusetts Medical Center...,single_hop_specifc_query_synthesizer
2,How does exercise help with mental health acco...,[Write letters to or from your future self Jou...,Exercise affects the brain in multiple benefic...,single_hop_specifc_query_synthesizer
3,What role do Licensed Professional Counselors ...,[social interactions How to set and maintain b...,Licensed Professional Counselors offer counsel...,single_hop_specifc_query_synthesizer
4,What are the healthy fats mentioned in the con...,[PART 2: NUTRITION AND DIET Chapter 4: Fundame...,"Healthy fats are from sources like olive oil, ...",single_hop_specifc_query_synthesizer
5,How can practcing self-compassion help in seti...,[<1-hop>\n\nsocial interactions How to set and...,Practicing self-compassion when setting and ma...,multi_hop_abstract_query_synthesizer
6,"How can mindfulness and meditation techniques,...",[<1-hop>\n\nPART 2: NUTRITION AND DIET Chapter...,Mindfulness and meditation practices are effec...,multi_hop_abstract_query_synthesizer
7,How can recognizing warning signs for mental h...,[<1-hop>\n\nThe Mental Health and Psychology H...,Recognizing warning signs for mental health is...,multi_hop_abstract_query_synthesizer
8,what chapter 9 and 16 tell about sleep and soc...,[<1-hop>\n\nPART 2: NUTRITION AND DIET Chapter...,Chapter 9 explains that good sleep is crucial ...,multi_hop_specific_query_synthesizer
9,How can social media impact mental health and ...,[<1-hop>\n\nsocial interactions How to set and...,Social media can significantly impact mental h...,multi_hop_specific_query_synthesizer


### Abstracted SDG

The above method is the full process - but we can shortcut that using the provided abstractions!

This will generate our knowledge graph under the hood, and will - from there - generate our personas and scenarios to construct our queries.



In [28]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_langchain_docs(docs, testset_size=10)

Applying HeadlinesExtractor:   0%|          | 0/2 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/2 [00:00<?, ?it/s]

Applying SummaryExtractor:   0%|          | 0/2 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/7 [00:00<?, ?it/s]

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/16 [00:00<?, ?it/s]

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/2 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/12 [00:00<?, ?it/s]

In [29]:
dataset.to_pandas()

,user_input,reference_contexts,reference,synthesizer_name
0,As a wellness coach dedicated to supporting in...,[The Mental Health and Psychology Handbook A P...,The context discusses mental health in the Uni...,single_hop_specifc_query_synthesizer
1,What is the significance of the University of ...,[PART 2: THERAPEUTIC APPROACHES Chapter 4: Cog...,The University of Massachusetts Medical Center...,single_hop_specifc_query_synthesizer
2,What does the context say about the relationsh...,[Write letters to or from your future self Jou...,The context does not explicitly mention dement...,single_hop_specifc_query_synthesizer
3,What are some effective mental health resource...,[social interactions How to set and maintain b...,The context mentions various mental health res...,single_hop_specifc_query_synthesizer
4,"How can mindfulness and meditation, as part of...",[<1-hop>\n\nPART 2: NUTRITION AND DIET Chapter...,The context explains that mindfulness is the p...,multi_hop_abstract_query_synthesizer
5,How does social connection and community engag...,[<1-hop>\n\nWrite letters to or from your futu...,The context highlights that strong social conn...,multi_hop_abstract_query_synthesizer
6,How can I use meal planning and strategies for...,[<1-hop>\n\nPART 2: NUTRITION AND DIET Chapter...,To create a healthy breakfast routine that sup...,multi_hop_abstract_query_synthesizer
7,How do stress and mental wellness relate to st...,[<1-hop>\n\nPART 2: NUTRITION AND DIET Chapter...,Stress is the body's response to demands or th...,multi_hop_abstract_query_synthesizer
8,How do Chapters 10 and 16 together inform stra...,[<1-hop>\n\nPART 2: NUTRITION AND DIET Chapter...,Chapters 10 and 16 provide complementary insig...,multi_hop_specific_query_synthesizer
9,How do Chapters 5 and 16 together inform a hol...,[<1-hop>\n\nPART 2: NUTRITION AND DIET Chapter...,Chapter 5 emphasizes effective meal planning a...,multi_hop_specific_query_synthesizer


## ❓ Question #2:

Ragas offers both an "unrolled" (manual) approach and an "abstracted" (automatic) approach to synthetic data generation. What are the trade-offs between these two approaches? When would you choose one over the other?

##### Answer:
Unrolled (manual)
You build the pipeline step by step:
1. Create and populate the KnowledgeGraph
2. Add documents as nodes (e.g. Node, NodeType.DOCUMENT)
3. Choose and apply transforms (e.g. default_transforms, HeadlineSplitter)
4. Save/load the knowledge graph for reuse
5. Create TestsetGenerator with your pre-built knowledge_graph
6. Set query_distribution (synthesizers and their weights)
7. Call generator.generate(testset_size, query_distribution)

Abstracted (automatic)
You use a single high-level call:
1. generator = TestsetGenerator(llm, embedding_model) (no knowledge_graph)
2. generator.generate_with_langchain_docs(docs, testset_size=10)
Knowledge graph creation, transforms, personas, scenarios, and query synthesis are handled internally.

Unrolled (manual) approach
Pros
1. Lets you control how the knowledge graph is built.
2. You can reuse the same knowledge graph across runs.
3. You can change transforms and query distribution to fit your needs.
Cons
1. Requires more code and setup.

Abstracted (automatic) approach
Pros
1. One call does everything.
2. Minimal setup and coding.
3. Good for quick experiments.
Cons
1. Uses built-in defaults, so you have little control.
2. The knowledge graph is not saved for later use.

When to choose which
Use unrolled (manual) when you:
    1. Need a custom query distribution (e.g. more multi-hop questions)
    2. Want to reuse the same KG across runs
    3. Use custom transforms for your domain
    4. Need to debug or inspect KG structure
    5. Are optimizing for evaluation quality and control

Use abstracted (automatic) when you:
    1. Want to quickly produce a test set
    2. Are fine with default transforms and query distribution
    3. Don’t need to persist the KG
    4. Are prototyping or experimenting
    5. Prefer minimal code over full control

---
## 🏗️ Activity #1: Custom Query Distribution

Modify the `query_distribution` to experiment with different ratios of query types.

### Requirements:
1. Create a custom query distribution with different weights than the default
2. Generate a new test set using your custom distribution
3. Compare the types of questions generated with the default distribution
4. Explain why you chose the weights you did

In [ ]:
### YOUR CODE HERE ###
'''
1. I created a custom query distribution with different weights than the default 0.4, 0.3, 0.3
3. The questions generated are more complex and require more information to answer.
4. I chose the weights 0.4, 0.3, 0.3 because I wanted to increase the number of multi-hop questions and decrease the number of single-hop questions.    
'''

# Define a custom query distribution with different weights
new_query_distribution = [
        (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.4),
        (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0.3),
        (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0.3),
]

# Generate a new test set and compare with the default
new_testset = generator.generate(testset_size=10, query_distribution=new_query_distribution)
custom_df = new_testset.to_pandas()

# Define default_df from testset (run the default generation cell above first!)
default_df = testset.to_pandas()

# Compare question types: default (0.5, 0.25, 0.25) vs custom (0.4, 0.3, 0.3)
print("Default distribution:")
print(default_df["synthesizer_name"].value_counts())
print("\nCustom distribution:")
print(custom_df["synthesizer_name"].value_counts())

new_testset.to_pandas()

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/10 [00:00<?, ?it/s]

Default distribution:
synthesizer_name
single_hop_specifc_query_synthesizer    5
multi_hop_abstract_query_synthesizer    3
multi_hop_specific_query_synthesizer    3
Name: count, dtype: int64

Custom distribution:
synthesizer_name
single_hop_specifc_query_synthesizer    4
multi_hop_abstract_query_synthesizer    3
multi_hop_specific_query_synthesizer    3
Name: count, dtype: int64


,user_input,reference_contexts,reference,synthesizer_name
0,"like, what is the psychology handbook thingy f...",[The Mental Health and Psychology Handbook A P...,The Mental Health and Psychology Handbook is a...,single_hop_specifc_query_synthesizer
1,What is the correct spelling of DBT?,[PART 2: THERAPEUTIC APPROACHES Chapter 4: Cog...,The context mentions the term as 'DBT' and ref...,single_hop_specifc_query_synthesizer
2,How does cortisol relate to stress management?,[Write letters to or from your future self Jou...,"Cortisol is a stress hormone, and managing its...",single_hop_specifc_query_synthesizer
3,What are some mental health resources availabl...,[social interactions How to set and maintain b...,The context mentions various mental health res...,single_hop_specifc_query_synthesizer
4,"How can improving sleep hygiene and routines, ...",[<1-hop>\n\nPART 2: NUTRITION AND DIET Chapter...,Improving sleep hygiene and routines promotes ...,multi_hop_abstract_query_synthesizer
5,how journaling practices and self-reflection h...,[<1-hop>\n\nWrite letters to or from your futu...,The context explains that journaling best prac...,multi_hop_abstract_query_synthesizer
6,How stress and mental wellness can be managed ...,[<1-hop>\n\nPART 2: NUTRITION AND DIET Chapter...,Stress and mental wellness can be improved by ...,multi_hop_abstract_query_synthesizer
7,How does the chapter 7 sleep science relate to...,[<1-hop>\n\nPART 2: NUTRITION AND DIET Chapter...,"Chapter 7 discusses the science of sleep, emph...",multi_hop_specific_query_synthesizer
8,How can implementing digital mental health str...,[<1-hop>\n\nsocial interactions How to set and...,Implementing digital mental health strategies ...,multi_hop_specific_query_synthesizer
9,How do Chapters 4 and 16 together inform a hol...,[<1-hop>\n\nPART 2: NUTRITION AND DIET Chapter...,Chapter 4 emphasizes the importance of a balan...,multi_hop_specific_query_synthesizer


We'll need to provide our LangSmith API key, and set tracing to "true".

---
# 🤝 Breakout Room #2
## RAG Evaluation with LangSmith

## Task 4: LangSmith Dataset

Now we can move on to creating a dataset for LangSmith!

First, we'll need to create a dataset on LangSmith using the `Client`!

We'll name our Dataset to make it easy to work with later.

In [65]:
from langsmith import Client
import uuid

client = Client()

dataset_name = f"Use Case Synthetic Data - AIE9 - {uuid.uuid4()}"

langsmith_dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Synthetic Data for Use Cases"
)

We'll iterate through the RAGAS created dataframe - and add each example to our created dataset!

> NOTE: We need to conform the outputs to the expected format - which in this case is: `question` and `answer`.

In [66]:
for data_row in dataset.to_pandas().iterrows():
  client.create_example(
      inputs={
          "question": data_row[1]["user_input"]
      },
      outputs={
          "answer": data_row[1]["reference"]
      },
      metadata={
          "context": data_row[1]["reference_contexts"]
      },
      dataset_id=langsmith_dataset.id
  )

## Basic RAG Chain

Time for some RAG!


In [67]:
rag_documents = docs

To keep things simple, we'll just use LangChain's recursive character text splitter!


In [68]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

We'll create our vectorstore using OpenAI's [`text-embedding-3-small`](https://platform.openai.com/docs/guides/embeddings/embedding-models) embedding model.

In [69]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

As usual, we will power our RAG application with Qdrant!

In [70]:
from langchain_qdrant import QdrantVectorStore

vectorstore = QdrantVectorStore.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="use_case_rag"
)

In [71]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

To get the "A" in RAG, we'll provide a prompt.

In [72]:
from langchain_core.prompts import ChatPromptTemplate

RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

Context: {context}
Question: {question}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_PROMPT)

As is usual: We'll be using `gpt-4.1-mini` for our RAG!

In [73]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-mini")

Finally, we can set-up our RAG LCEL chain!

In [74]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.output_parsers import StrOutputParser

rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | rag_prompt | llm | StrOutputParser()
)

In [75]:
rag_chain.invoke({"question" : "What are some recommended exercises for lower back pain?"})

'Recommended exercises for lower back pain include:\n\n- Cat-Cow Stretch: Start on hands and knees, alternate between arching your back up (cat) and letting it sag down (cow). Do 10-15 repetitions.  \n- Bird Dog: From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold for 5 seconds, then switch sides. Do 10 repetitions per side.  \n- Partial Crunches: Lie on your back with knees bent, cross arms over chest, tighten stomach muscles and raise shoulders off floor. Hold briefly, then lower. Do 8-12 repetitions.  \n- Knee-to-Chest Stretch: Lie on your back, pull one knee toward your chest while keeping the other foot flat. Hold for 15-30 seconds, then switch legs.  \n- Pelvic Tilts: Lie on your back with knees bent, flatten your back against the floor by tightening abs and tilting pelvis up slightly. Hold for 10 seconds, repeat 8-12 times.'

## LangSmith Evaluation Set-up

We'll use OpenAI's GPT-4.1 as our evaluation LLM for our base Evaluators.

In [76]:
eval_llm = ChatOpenAI(model="gpt-4.1")

We'll be using a number of evaluators - from LangSmith provided evaluators, to a few custom evaluators!

In [77]:
from openevals.llm import create_llm_as_judge
from langsmith.evaluation import evaluate

# 1. QA Correctness (replaces LangChainStringEvaluator("qa"))
qa_evaluator = create_llm_as_judge(
    prompt="You are evaluating a QA system. Given the input, assess whether the prediction is correct.\n\nInput: {inputs}\nPrediction: {outputs}\nReference answer: {reference_outputs}\n\nIs the prediction correct? Return 1 if correct, 0 if incorrect.",
    feedback_key="qa",
    model="openai:gpt-4o" ,  # pass your LangChain chat model directly
)

# 2. Labeled Helpfulness (replaces LangChainStringEvaluator("labeled_criteria"))
labeled_helpfulness_evaluator = create_llm_as_judge(
    prompt=(
        "You are assessing a submission based on the following criterion:\n\n"
        "helpfulness: Is this submission helpful to the user, "
        "taking into account the correct reference answer?\n\n"
        "Input: {inputs}\n"
        "Submission: {outputs}\n"
        "Reference answer: {reference_outputs}\n\n"
        "Does the submission meet the criterion? Return 1 if yes, 0 if no."
    ),
    feedback_key="helpfulness",
    model="openai:gpt-4o" ,
)

# 3. Dopeness (replaces LangChainStringEvaluator("criteria"))
dopeness_evaluator = create_llm_as_judge(
    prompt=(
        "You are assessing a submission based on the following criterion:\n\n"
        "dopeness: Is this response dope, lit, cool, or is it just a generic response?\n\n"
        "Input: {inputs}\n"
        "Submission: {outputs}\n\n"
        "Does the submission meet the criterion? Return 1 if yes, 0 if no."
    ),
    feedback_key="dopeness",
    model="openai:gpt-4o" ,
)

> **Describe what each evaluator is evaluating:**
>
> - `qa_evaluator`:
> - `labeled_helpfulness_evaluator`:
> - `dopeness_evaluator`:

## LangSmith Evaluation

In [78]:
evaluate(
    rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        dopeness_evaluator
    ],
    metadata={"revision_id": "default_chain_init"},
)

View the evaluation results for experiment: 'weary-grade-66' at:
https://smith.langchain.com/o/b965f3f5-feac-4cd8-9871-11c924ce3a82/datasets/4f1bdd0d-04d0-4356-94ff-f9bf2194daa5/compare?selectedSessions=8dfcf7a2-bcf0-421a-bfcf-f21c04b016cf




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.qa,feedback.helpfulness,feedback.dopeness,execution_time,example_id,id
0,How does Chapter 4 relate to Chapter 2 in prom...,"Based on the provided context, Chapter 4 (""Fun...",None,Chapter 4 focuses on the fundamentals of healt...,True,True,False,2.889361,aeda7a30-afc9-4cfe-a5f5-59ca02cd1444,019c5afe-6245-7160-8a6e-2f7b1aa5df43
1,How do the foundational principles of healthy ...,"Based on the provided context, Chapter 4 discu...",None,Chapter 4 emphasizes the importance of a balan...,True,True,True,2.542369,1490407d-92cd-4052-b5c6-f2dd3cfc0b6b,019c5afe-afed-7142-b93d-3a6186e85222
2,How do Chapters 5 and 16 together inform a hol...,Chapters 5 and 16 together inform a holistic a...,None,Chapter 5 emphasizes effective meal planning a...,True,True,True,3.400355,1dadd4c1-7eaa-4087-8af2-170eaf423729,019c5afe-ecf9-7443-b760-4909000110b7
3,How do Chapters 10 and 16 together inform stra...,Chapters 10 and 16 together highlight the rela...,None,Chapters 10 and 16 provide complementary insig...,True,True,True,3.451199,bf87bff7-13ee-4cd3-b829-44e4b59fbcb9,019c5aff-28c8-7d12-a6df-b3b999b00e77
4,How do stress and mental wellness relate to st...,Stress and mental wellness are closely related...,None,Stress is the body's response to demands or th...,True,True,False,2.736951,00f6a17f-1194-4070-9bd0-3dc4b13e7c45,019c5aff-6c45-7323-ac75-2502efc6e2e0
5,How can I use meal planning and strategies for...,"Based on the context, you can create a healthy...",None,To create a healthy breakfast routine that sup...,True,True,True,4.607411,c04e6313-93ac-4801-a78a-13728c8175a3,019c5aff-a50f-7b30-855c-db99487d4d85
6,How does social connection and community engag...,Social connection and community engagement imp...,None,The context highlights that strong social conn...,True,True,True,2.719577,7fdffb0a-0774-4a47-8356-7a9b7f8981f1,019c5aff-e6d5-7c41-8150-51b249a075ac
7,"How can mindfulness and meditation, as part of...","According to the context, mindfulness and medi...",None,The context explains that mindfulness is the p...,True,True,False,2.253682,15d0ebfb-c0d8-4c10-98ed-cd3ba1fd5bcb,019c5b00-2842-7cb0-9eea-9f353b6810e0
8,What are some effective mental health resource...,I don't know.,None,The context mentions various mental health res...,False,False,False,0.993220,59eb1a0a-3389-4d27-a802-390ab67485d3,019c5b00-53db-7181-bc84-f5ef29d16801
9,What does the context say about the relationsh...,The context states that exercise improves cogn...,None,The context does not explicitly mention dement...,False,False,False,0.817690,c4838615-0a13-4457-97d0-2b9c707ceba3,019c5b00-79da-77a3-9f57-a3e1efad0e85


## Dope-ifying Our Application

We'll be making a few changes to our RAG chain to increase its performance on our SDG evaluation test dataset!

- Include a "dope" prompt augmentation
- Use larger chunks
- Improve the retriever model to: `text-embedding-3-large`

Let's see how this changes our evaluation!

In [80]:
DOPENESS_RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

Make your answer rad, ensure high levels of dopeness. Do not be generic, or give generic responses.

Context: {context}
Question: {question}
"""

dopeness_rag_prompt = ChatPromptTemplate.from_template(DOPENESS_RAG_PROMPT)

In [81]:
rag_documents = docs

In [82]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

## ❓ Question #3:

Why would modifying our chunk size modify the performance of our application?

##### Answer:
Chunk size affects both retrieval and context that the LLM sees, which in turn affects answer quality.

1. Retrieval quality
Smaller chunks: More precise matches per chunk, but a single relevant passage can be split across several chunks. The model may only see part of the needed information.
Larger chunks: Each chunk can carry more complete context, but retrieval may return less relevant portions of the chunk (e.g., only one paragraph in a long chunk is relevant).

2. Context window usage
Smaller chunks → more chunks for a given top_k → more fragments to piece together, with more risk of redundancy or missing connections.
Larger chunks → fewer chunks, but each one uses more of the LLM context window, possibly squeezing out other relevant chunks.

3. Multi-hop and reasoning
In the notebook you use MultiHopSpecificQuerySynthesizer, which asks questions that need information from multiple places.
    a. If chunks are too small, important connections between ideas may be split across chunks.
    b. If chunks are too large, retrieval may not surface all the right segments of text.


In [86]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

## ❓ Question #4:

Why would modifying our embedding model modify the performance of our application?

##### Answer:
1. Different vector spaces
Each embedding model builds its own representation of meaning:
a. Vectors live in a model-specific space (e.g., 384 vs 1536 dimensions)
b. Words and sentences that are “similar” to one model may be “different” to another
c. Models are trained on different data and objectives, so their notion of similarity varies

2. Retrieval depends on similarity
In a RAG-style setup:
a. Documents and queries are embedded
b. Retrieval finds documents whose vectors are nearest to the query vector
c. Changing the model changes which documents are considered similar
d. The same query can return different documents under different models

3. Index compatibility
a. The vector index is built with embeddings from model A
b. If you switch to model B and only re-embed queries, you’re comparing vectors from two different spaces
c. That comparison is not meaningful; you must re-embed all documents (and rebuild the index) when changing models

4. Different strengths
Different models do better on different tasks, for example:
a. Multilingual vs. English-only
b. Longer vs. shorter texts
c. Specific domains (e.g., legal, medical) vs. general text
Switching models can improve or worsen performance depending on how well the new model matches your use case.



In [84]:
from langchain_qdrant import QdrantVectorStore

vectorstore = QdrantVectorStore.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="Use Case RAG Docs"
)

In [87]:
retriever = vectorstore.as_retriever()

Setting up our new and improved DOPE RAG CHAIN.

In [88]:
dopeness_rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | dopeness_rag_prompt | llm | StrOutputParser()
)

Let's test it on the same output that we saw before.

In [89]:
dopeness_rag_chain.invoke({"question" : "How can I improve my sleep quality?"})

'Alright, brace yourself for some next-level sleep wizardry straight from the science-backed vault! To crank your sleep quality to max dopeness, here’s the ultimate playbook rooted in that rad context:\n\n1. **Lock Down a Consistent Sleep Schedule** – Be the boss of your bedtime and wake-up time, every day, even weekends. Your body LOVES rhythm, and this groove gets your internal clock synced up like a pro DJ spinning tracks.\n\n2. **Craft a Legendary Bedtime Ritual** – Think chill vibes only: reading a killer book, gentle stretching moves, or a warm bath that melts away the day’s stress. This tells your brain, “Yo, it’s time to drop in and zone out.”\n\n3. **Perfect Your Sleep Dungeon** – Keep that bedroom cool (65-68°F / 18-20°C). Blackout curtains or a sleep mask are your stealthy allies against sneaky light. Silence? White noise machines or earplugs crush distractions. Also, invest in a mattress and pillows so comfy, it’s like sleeping on clouds.\n\n4. **Screen Time Shutdown** – Di

Finally, we can evaluate the new chain on the same test set!

In [90]:
evaluate(
    dopeness_rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        dopeness_evaluator
    ],
    metadata={"revision_id": "dopeness_rag_chain"},
)

View the evaluation results for experiment: 'reflecting-test-73' at:
https://smith.langchain.com/o/b965f3f5-feac-4cd8-9871-11c924ce3a82/datasets/4f1bdd0d-04d0-4356-94ff-f9bf2194daa5/compare?selectedSessions=7ebca17c-f04d-4909-a525-b0463b21fa32




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.qa,feedback.helpfulness,feedback.dopeness,execution_time,example_id,id
0,How does Chapter 4 relate to Chapter 2 in prom...,"Alright, let’s crank the wellness dial to max ...",None,Chapter 4 focuses on the fundamentals of healt...,True,True,True,3.543513,aeda7a30-afc9-4cfe-a5f5-59ca02cd1444,019c5b02-a0d0-7703-8021-e68c4872308c
1,How do the foundational principles of healthy ...,"Alright, let’s crank up the wellness wisdom to...",None,Chapter 4 emphasizes the importance of a balan...,True,True,True,3.560720,1490407d-92cd-4052-b5c6-f2dd3cfc0b6b,019c5b02-d65a-78a2-a030-5d16104b1cf5
2,How do Chapters 5 and 16 together inform a hol...,"Alright, let’s blend the wisdom from Chapters ...",None,Chapter 5 emphasizes effective meal planning a...,True,True,True,3.681395,1dadd4c1-7eaa-4087-8af2-170eaf423729,019c5b03-0b78-7aa1-ab10-797227ca2e03
3,How do Chapters 10 and 16 together inform stra...,"Alright, let’s dive deep into this digital zen...",None,Chapters 10 and 16 provide complementary insig...,True,True,True,5.320949,bf87bff7-13ee-4cd3-b829-44e4b59fbcb9,019c5b03-4512-73e1-928f-dceb8be1679b
4,How do stress and mental wellness relate to st...,"Alright, let’s break down the rad connection b...",None,Stress is the body's response to demands or th...,True,True,True,5.427291,00f6a17f-1194-4070-9bd0-3dc4b13e7c45,019c5b03-7eaa-72b0-92e4-9b1effae99e0
5,How can I use meal planning and strategies for...,"Alright, let’s crank your breakfast routine up...",None,To create a healthy breakfast routine that sup...,True,True,True,6.198300,c04e6313-93ac-4801-a78a-13728c8175a3,019c5b03-afa7-74f0-baa8-903c3438d9b7
6,How does social connection and community engag...,"Alright, let’s peel back the layers and drop s...",None,The context highlights that strong social conn...,True,True,True,5.034197,7fdffb0a-0774-4a47-8356-7a9b7f8981f1,019c5b03-e6a9-7150-9458-359ade090925
7,"How can mindfulness and meditation, as part of...","Alright, buckle up for a mind-expanding ride i...",None,The context explains that mindfulness is the p...,True,True,True,4.123198,15d0ebfb-c0d8-4c10-98ed-cd3ba1fd5bcb,019c5b04-1fdc-7702-b711-30f628edf015
8,What are some effective mental health resource...,"Yo, here’s the scoop straight from the brain v...",None,The context mentions various mental health res...,False,True,True,4.270132,59eb1a0a-3389-4d27-a802-390ab67485d3,019c5b04-4fde-7e41-aaff-0b109dd4c42d
9,What does the context say about the relationsh...,I gotta keep it 100 with you—the context drops...,None,The context does not explicitly mention dement...,True,True,True,1.232150,c4838615-0a13-4457-97d0-2b9c707ceba3,019c5b04-87da-73e1-bcfb-047e5067a4c6


---
## 🏗️ Activity #2: Analyze Evaluation Results

Provide a screenshot of the difference between the two chains in LangSmith, and explain why you believe certain metrics changed in certain ways.

##### Answer:
Screenshot is added in repo, same directory : LangSmith_result_screenshot.png


The difference in two runs is dopeness_rag_prompt. This enables the answer to be very specific rad and dope. That is seen in the result where dopeness and helpfulness is 100% and qa is 92%. 

---
## Summary

In this session, we:

1. **Generated synthetic test data** using Ragas' knowledge graph-based approach
2. **Explored query synthesizers** for creating diverse question types
3. **Loaded synthetic data** into a LangSmith dataset for evaluation
4. **Built and evaluated a RAG chain** using LangSmith evaluators
5. **Iterated on the pipeline** by modifying chunk size, embedding model, and prompt — then measured the impact

### Key Takeaways:

- **Synthetic data generation** is critical for early iteration — it provides high-quality signal without manually creating test data
- **LangSmith evaluators** enable systematic comparison of pipeline versions
- **Small changes matter** — chunk size, embedding model, and prompt modifications can significantly affect evaluation scores